# <center style="font-family: consolas; font-size: 32px; font-weight: bold;">  Hands-On LangChain for LLM Applications Development: Chatbots Memory </center>

# <center style="font-family: consolas; font-size: 25px; font-weight: bold;">  (OpenRouter + Google Colab + LangChain 1.x / LangGraph Edition) </center>
***

When interacting with language models, such as Chatbots, the absence of memory poses a significant hurdle in creating natural and seamless conversations. Users expect continuity and context retention, which traditional models lack. This limitation becomes particularly evident in applications where ongoing dialogue is crucial for user engagement and satisfaction.

LangChain offers robust solutions to address this challenge. Memory, in this context, refers to the ability of the language model to remember previous parts of a conversation and use that information to inform subsequent interactions.

**Important — this is a full modernization, not just an OpenRouter swap.** The original version of this notebook used `ConversationChain`, `ConversationBufferMemory`, `ConversationBufferWindowMemory`, `ConversationTokenBufferMemory`, and `ConversationSummaryBufferMemory`. **All of these classes are deprecated / removed in modern LangChain (1.x)** — importing them today either raises an `ImportError` or a hard deprecation warning telling you to migrate.

The officially recommended replacement is **LangGraph**, LangChain's own graph/state library, which is now the standard way to give a chatbot persistent, multi-turn memory. This notebook rebuilds every section using the modern equivalents:

| Old (deprecated) | Modern replacement |
|---|---|
| `ConversationChain` + `ConversationBufferMemory` | `StateGraph` + `MessagesState` + a checkpointer (`MemorySaver`) |
| `ConversationBufferWindowMemory(k=...)` | `trim_messages(strategy="last", token_counter=len, ...)` inside the graph node |
| `ConversationTokenBufferMemory(max_token_limit=...)` | `trim_messages(strategy="last", token_counter=<real token counter>, max_tokens=...)` |
| `ConversationSummaryBufferMemory` | A custom LangGraph node that summarizes older turns with the LLM once a token threshold is passed |

#### <a id="top"></a>
# <div style="box-shadow: rgb(60, 121, 245) 0px 0px 0px 3px inset, rgb(255, 255, 255) 10px -10px 0px -3px, rgb(31, 193, 27) 10px -10px, rgb(255, 255, 255) 20px -20px 0px -3px, rgb(255, 217, 19) 20px -20px, rgb(255, 255, 255) 30px -30px 0px -3px, rgb(255, 156, 85) 30px -30px, rgb(255, 255, 255) 40px -40px 0px -3px, rgb(255, 85, 85) 40px -40px; padding:20px; margin-right: 40px; font-size:30px; font-family: consolas; text-align:center; display:fill; border-radius:15px; color:rgb(60, 121, 245);"><b>Table of contents</b></div>

<div style="background-color: rgba(60, 121, 245, 0.03); padding:30px; font-size:15px; font-family: consolas;">
<ul>
    <li><a href="#0" target="_self" rel=" noreferrer nofollow">0. Setting Up Working Environment with OpenRouter on Colab </a> </li>
    <li><a href="#1" target="_self" rel=" noreferrer nofollow">1. Basic Memory with LangGraph (replaces ConversationBufferMemory) </a></li>
    <li><a href="#2" target="_self" rel=" noreferrer nofollow">2. Windowed Memory (replaces ConversationBufferWindowMemory) </a></li>
    <li><a href="#3" target="_self" rel=" noreferrer nofollow">3. Token-Limited Memory (replaces ConversationTokenBufferMemory) </a></li>
    <li><a href="#4" target="_self" rel=" noreferrer nofollow">4. Summary Memory (replaces ConversationSummaryBufferMemory) </a></li>
</ul>
</div>

***


<a id="0"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 0. Setting Up Working Environment with OpenRouter on Colab </b></div>

Let's install the required libraries first, then load our OpenRouter API key and set up the client. We now install `langgraph` too, since it's the engine behind our memory implementations.


In [1]:
# Install required libraries (Colab)
!pip install -q openai langchain langchain-core langchain-openai langgraph


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 19.6 MB/s eta 0:00:00


### OpenRouter API Key

Create an account at [openrouter.ai](https://openrouter.ai/keys) and get an API key.

On Colab, store the key in **Secrets** (the 🔑 icon on the left sidebar) under the name `OPENROUTER_API_KEY`. If the secret isn't found, you'll be prompted to enter it manually.


In [2]:
import os

try:
    # If you're running this on Google Colab
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    OPENROUTER_API_KEY = None

if not OPENROUTER_API_KEY:
    import getpass
    OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY


In [3]:
from langchain_openai import ChatOpenAI

# Pick any model available on OpenRouter: https://openrouter.ai/models
llm_model = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    temperature=0.0,
    model=llm_model,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
)


<a id="1"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 1. Basic Memory with LangGraph </b></div>


Let's start with a motivating example for memory, using LangGraph to manage a chat/chatbot conversation. The old `ConversationChain` + `ConversationBufferMemory` combo is replaced by:

1. A **`StateGraph`** whose state is `MessagesState` (just a growing list of messages).
2. A single **node** that calls the LLM with the full message history.
3. A **checkpointer** (`MemorySaver`) that automatically persists the message list per conversation, keyed by a `thread_id` — this *is* the memory.


In [4]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langchain_core.messages import HumanMessage

# Define the graph
workflow = StateGraph(state_schema=MessagesState)

def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": response}

workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

# The checkpointer is what gives us memory across calls
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)


Every call now needs a `thread_id` inside the `config` — this is how LangGraph knows *which* conversation's memory to load and append to (equivalent to having one `ConversationBufferMemory` instance per user/session).

Let's start a conversation with the input "Hi, my name is Youssef" and see the response.


In [5]:
config = {"configurable": {"thread_id": "conversation-1"}}

output = app.invoke(
    {"messages": [HumanMessage(content="Hi, my name is Youssef")]},
    config,
)
print(output["messages"][-1].content)


Hi Youssef! How can I assist you today?


Then, let's ask it what 1 + 1 is.


In [6]:
output = app.invoke(
    {"messages": [HumanMessage(content="What is 1+1?")]},
    config,
)
print(output["messages"][-1].content)


1 + 1 equals 2.


Now we will ask it again "What's my name?" — and because we reused the same `thread_id`, LangGraph automatically loaded the full prior history before calling the model, so it should still know.


In [7]:
output = app.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config,
)
print(output["messages"][-1].content)


Your name is Youssef.


To inspect everything that's been remembered so far (the equivalent of `memory.buffer` / `memory.load_memory_variables({})`), you can pull the graph's current state for that `thread_id`:


In [8]:
state = app.get_state(config)
for m in state.values["messages"]:
    print(f"{m.type}: {m.content}")


human: Hi, my name is Youssef
ai: Hi Youssef! How can I assist you today?
human: What is 1+1?
ai: 1 + 1 equals 2.
human: What is my name?
ai: Your name is Youssef.


If you want to seed a conversation's memory manually (the equivalent of the old `memory.save_context({"input": "Hi"}, {"output": "What's up"})`), you can invoke the graph with both a `HumanMessage` and directly append an `AIMessage`, or simply update the checkpointer's state:


In [9]:
from langchain_core.messages import AIMessage

seed_config = {"configurable": {"thread_id": "seeded-conversation"}}

app.update_state(
    seed_config,
    {"messages": [HumanMessage(content="Hi"), AIMessage(content="What's up")]},
)

state = app.get_state(seed_config)
for m in state.values["messages"]:
    print(f"{m.type}: {m.content}")


human: Hi
ai: What's up


When you use a large language model for a chat conversation, the large language model itself is stateless — it does not remember the conversation you've had so far. Each call to the API endpoint is independent. Chatbots have memory only because the surrounding application code (here, LangGraph + its checkpointer) resends the full conversation history as context on every call.

As the conversation becomes long, the amount of history sent grows, and since most LLM providers charge per token, this gets more expensive. So — just like the original LangChain memory classes — we need ways to cap how much history gets sent. Let's look at the modern equivalents.


<a id="2"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 2. Windowed Memory </b></div>


The old `ConversationBufferWindowMemory(k=1)` only kept the most recent `k` exchanges. The modern equivalent is `trim_messages(...)` from `langchain_core.messages`, applied *inside* the graph node before calling the LLM — using `token_counter=len` makes it count **messages** instead of tokens, replicating a "window."

Let's build a version that only keeps the last 2 messages (i.e. `k=1`: one human turn + one AI turn).


In [10]:
from langchain_core.messages import trim_messages

window_trimmer = trim_messages(
    max_tokens=2,          # counting messages, not tokens, since token_counter=len
    strategy="last",
    token_counter=len,
    include_system=True,
    allow_partial=False,
)

def call_model_windowed(state: MessagesState):
    trimmed = window_trimmer.invoke(state["messages"])
    response = llm.invoke(trimmed)
    return {"messages": response}

window_workflow = StateGraph(state_schema=MessagesState)
window_workflow.add_edge(START, "model")
window_workflow.add_node("model", call_model_windowed)

window_memory = MemorySaver()
window_app = window_workflow.compile(checkpointer=window_memory)


Let's rerun the same conversation as before: "Hi, my name is Youssef", then "What is 1+1?", then "What is my name?". Because we're only keeping the last 2 messages before each call, the model should have forgotten the introduction by the time we ask for the name.


In [11]:
window_config = {"configurable": {"thread_id": "windowed-conversation"}}

for user_input in ["Hi, my name is Youssef", "What is 1+1?", "What is my name?"]:
    output = window_app.invoke({"messages": [HumanMessage(content=user_input)]}, window_config)
    print(f"User: {user_input}")
    print(f"AI: {output['messages'][-1].content}\n")


User: Hi, my name is Youssef
AI: Hi Youssef! How can I assist you today?

User: What is 1+1?
AI: 1 + 1 equals 2.

User: What is my name?
AI: I don't have access to personal data about users unless it has been shared with me in the course of our conversation. Therefore, I don't know your name. If you'd like to share it, feel free!



Just like the original example, this shows the trade-off: keeping a small window saves tokens and cost, but the model loses access to earlier context (like the name mentioned in the first turn). In practice you'd use a larger `k` (i.e. a bigger `max_tokens` count here) than 1.


<a id="3"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 3. Token-Limited Memory </b></div>


With the old `ConversationTokenBufferMemory`, the memory limited the number of *tokens* saved rather than the number of messages — this maps more directly to the actual cost of LLM calls. We get the same behavior from `trim_messages` by swapping `token_counter=len` for a real tokenizer-based counter, e.g. the chat model's own `get_num_tokens_from_messages`.

Let's set the max token limit to 50 and feed in: "AI is what? Amazing! Backpropagation is what? Beautiful! Chatbots are what? Charming!"


In [13]:
import tiktoken
from langchain_core.messages import HumanMessage, AIMessage

def tiktoken_token_counter(messages):
    # This specific model might not be in tiktoken's default mapping, so we use 'cl100k_base'
    # which is common for newer OpenAI models like gpt-4o-mini.
    # For precise counting, it's best to verify the exact encoding for gpt-4o-mini if it becomes available.
    try:
        encoding = tiktoken.encoding_for_model(llm_model)
    except KeyError:
        # Fallback for models not explicitly in tiktoken's registry, often common for gpt-4 family
        encoding = tiktoken.get_encoding("cl100k_base")

    num_tokens = 0
    for message in messages:
        # According to OpenAI's documentation, each message adds a few tokens for structure
        # (e.g., role, content fields). A common heuristic is 4 tokens per message.
        num_tokens += 4
        if isinstance(message, (HumanMessage, AIMessage)): # Handle HumanMessage and AIMessage
            num_tokens += len(encoding.encode(message.content))
        # You might need to add handling for other message types if they are used
        # e.g., if isinstance(message, SystemMessage): num_tokens += len(encoding.encode(message.content))
    return num_tokens

token_trimmer_50 = trim_messages(
    max_tokens=50,
    strategy="last",
    token_counter=tiktoken_token_counter,     # Use the custom token counter
    include_system=True,
    allow_partial=False,
)

messages = [
    HumanMessage(content="AI is what?!"),
    AIMessage(content="Amazing!"),
    HumanMessage(content="Backpropagation is what?"),
    AIMessage(content="Beautiful!"),
    HumanMessage(content="Chatbots are what?"),
    AIMessage(content="Charming!"),
]

trimmed_50 = token_trimmer_50.invoke(messages)
for m in trimmed_50:
    print(f"{m.type}: {m.content}")

human: AI is what?!
ai: Amazing!
human: Backpropagation is what?
ai: Beautiful!
human: Chatbots are what?
ai: Charming!


If we run this with a high token limit, it keeps almost (or all of) the conversation. Let's bump the limit up to 100 tokens — now it should comfortably fit the whole exchange.


In [15]:
token_trimmer_100 = trim_messages(
    max_tokens=100,
    strategy="last",
    token_counter=tiktoken_token_counter, # Changed from `llm` to `tiktoken_token_counter`
    include_system=True,
    allow_partial=False,
)

trimmed_100 = token_trimmer_100.invoke(messages)
for m in trimmed_100:
    print(f"{m.type}: {m.content}")

human: AI is what?!
ai: Amazing!
human: Backpropagation is what?
ai: Beautiful!
human: Chatbots are what?
ai: Charming!


If we decrease the limit to 20 tokens, it chops off the earlier parts of the conversation to retain only the most recent exchanges that fit within the token budget.


In [17]:
token_trimmer_20 = trim_messages(
    max_tokens=20,
    strategy="last",
    token_counter=tiktoken_token_counter,
    include_system=True,
    allow_partial=False,
)

trimmed_20 = token_trimmer_20.invoke(messages)
for m in trimmed_20:
    print(f"{m.type}: {m.content}")

human: Chatbots are what?
ai: Charming!


Exactly like the windowed example, you'd plug this `trim_messages(...)` call into the LangGraph node (in place of `window_trimmer`) to get a live chatbot with token-capped memory.


<a id="4"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 4. Summary Memory </b></div>


Finally, there's the technique the old `ConversationSummaryBufferMemory` provided: instead of just dropping old messages once a token limit is hit, use the LLM itself to write a running **summary** of the older parts of the conversation, and keep that summary instead of the raw text.

There's no drop-in class for this anymore — the modern, officially documented pattern is a **custom LangGraph node** that: (1) checks whether the conversation has grown past a token threshold, and if so, (2) asks the LLM to summarize everything except the most recent messages, then (3) replaces the old messages with a single summary `SystemMessage`.

Here's an example, using the same "meeting schedule" text as before:


In [18]:
# create a long string
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."


In [22]:
from langchain_core.messages import SystemMessage, RemoveMessage
from typing import TypedDict


class SummaryState(MessagesState):
    summary: str


def call_model_with_summary(state: SummaryState):
    summary = state.get("summary", "")
    if summary:
        system_message = SystemMessage(content=f"Summary of earlier conversation: {summary}")
        messages_to_send = [system_message] + state["messages"]
    else:
        messages_to_send = state["messages"]

    response = llm.invoke(messages_to_send)
    return {"messages": response}


def summarize_conversation(state: SummaryState):
    summary = state.get("summary", "")
    if summary:
        summary_prompt = (
            f"This is the summary of the conversation so far: {summary}\n\n"
            "Extend the summary by taking into account the new messages above:"
        )
    else:
        summary_prompt = "Create a summary of the conversation above:"

    messages_for_summary = state["messages"] + [HumanMessage(content=summary_prompt)]
    summary_response = llm.invoke(messages_for_summary)

    # Keep only the last 2 messages, drop the rest (they're now captured in the summary)
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": summary_response.content, "messages": delete_messages}


def should_summarize(state: SummaryState):
    # Once we've accumulated too many messages, trigger a summary + prune
    if tiktoken_token_counter(state["messages"]) > 100: # Changed from llm.get_num_tokens_from_messages to tiktoken_token_counter
        return "summarize_conversation"
    return "__end__"

Now let's wire these into a graph with a conditional edge: after every model call, check `should_summarize` — if the running history is too big, route to the summarization node, which condenses everything except the last couple of turns.


In [23]:
from langgraph.graph import END

summary_workflow = StateGraph(state_schema=SummaryState)

summary_workflow.add_node("model", call_model_with_summary)
summary_workflow.add_node("summarize_conversation", summarize_conversation)

summary_workflow.add_edge(START, "model")
summary_workflow.add_conditional_edges(
    "model",
    should_summarize,
    {"summarize_conversation": "summarize_conversation", "__end__": END},
)
summary_workflow.add_edge("summarize_conversation", END)

summary_memory = MemorySaver()
summary_app = summary_workflow.compile(checkpointer=summary_memory)


We'll insert a few conversational turns, ending with "What is on the schedule today?", whose answer is the long `schedule` string above. Because the total token count will cross our 100-token threshold, the graph should automatically kick off a summary and prune the raw messages.


In [24]:
summary_config = {"configurable": {"thread_id": "summary-conversation"}}

for user_input, canned_output in [
    ("Hello", "What's up"),
    ("Not much, just hanging", "Cool"),
]:
    summary_app.invoke({"messages": [HumanMessage(content=user_input)]}, summary_config)

output = summary_app.invoke(
    {"messages": [HumanMessage(content="What is on the schedule today?")]},
    summary_config,
)
print(output["messages"][-1].content)


I don’t have access to real-time schedules or personal calendars, but I can help you plan your day or suggest activities! What do you have in mind?


Let's check the graph's internal state: notice `summary` now holds a condensed version of the earlier turns, while `messages` only keeps the most recent couple of exchanges — exactly the behavior `ConversationSummaryBufferMemory` used to give us, but built from transparent, inspectable building blocks instead of a single opaque memory class.


In [25]:
state = summary_app.get_state(summary_config)
print("Summary so far:")
print(state.values.get("summary", "(none yet)"))
print("\nRaw messages kept:")
for m in state.values["messages"]:
    print(f"{m.type}: {m.content[:80]}")


Summary so far:
In the conversation, the user greeted me and mentioned they were just hanging out. I responded by asking if there was anything specific on their mind or if they wanted to chat about something. The user then inquired about the schedule for the day, to which I explained that I don't have access to real-time schedules but offered to help plan their day or suggest activities.

Raw messages kept:
human: What is on the schedule today?
ai: I don’t have access to real-time schedules or personal calendars, but I can help


If we continue the conversation from here (e.g. "What would be a good demo to show?"), the model will rely on the `summary` system message we constructed, plus whatever raw messages remain, instead of the entire raw history — capping cost while still preserving the gist of everything that came before.


In [26]:
output = summary_app.invoke(
    {"messages": [HumanMessage(content="What would be a good demo to show?")]},
    summary_config,
)
print(output["messages"][-1].content)


It depends on your audience and the context of the demo, but here are a few ideas:

1. **Product Demo**: If you're showcasing a product, highlight its key features and benefits. Use real-life scenarios to demonstrate how it solves a problem.

2. **Software Walkthrough**: For a software application, guide users through the main functionalities, showing how to navigate the interface and complete common tasks.

3. **Interactive Presentation**: Use tools like Prezi or PowerPoint to create an engaging presentation that includes videos, animations, and interactive elements.

4. **Live Coding Session**: If you're in a tech environment, a live coding demo can be effective. Solve a problem or build a small application in real-time.

5. **Case Study**: Present a case study that illustrates the success of your product or service, including metrics and testimonials.

6. **Comparison Demo**: Compare your product with competitors to highlight unique features and advantages.

Let me know if you need 

***

LangChain also encompasses additional memory types beyond what we've covered here — most notably **vector store memory** (retrieving the most semantically relevant past messages via embeddings, rather than the most recent ones) and **entity memory** (tracking facts about specific people/things mentioned across a conversation). Both are commonly implemented today as custom LangGraph nodes/tools rather than built-in memory classes, following the same pattern shown above: state lives in the graph, and you write small, explicit functions to decide what gets kept, summarized, or retrieved.

### Summary of changes made in this version

- Replaced `kaggle_secrets` and direct OpenAI calls with **OpenRouter** (`base_url = https://openrouter.ai/api/v1`), following the Colab Secrets pattern used throughout this series.
- Replaced the deprecated `ConversationChain` + `ConversationBufferMemory` with a **LangGraph `StateGraph`** using `MessagesState` and a `MemorySaver` checkpointer keyed by `thread_id`.
- Replaced `ConversationBufferWindowMemory(k=...)` with **`trim_messages(strategy="last", token_counter=len, max_tokens=k*2)`**, applied inside the graph's model node.
- Replaced `ConversationTokenBufferMemory(max_token_limit=...)` with **`trim_messages(strategy="last", token_counter=llm, max_tokens=...)`**, using the chat model's real tokenizer.
- Replaced `ConversationSummaryBufferMemory` with a **custom summarization node** — a conditional edge checks the token count, and once exceeded, an LLM call condenses older turns into a running `summary` field while pruning the raw messages via `RemoveMessage`.

# <div style="box-shadow: rgba(240, 46, 170, 0.4) -5px 5px inset, rgba(240, 46, 170, 0.3) -10px 10px inset, rgba(240, 46, 170, 0.2) -15px 15px inset, rgba(240, 46, 170, 0.1) -20px 20px inset, rgba(240, 46, 170, 0.05) -25px 25px inset; padding:20px; font-size:30px; font-family: consolas; display:fill; border-radius:15px; color: rgba(240, 46, 170, 0.7)"> <b> ༼⁠ ⁠つ⁠ ⁠◕⁠‿⁠◕⁠ ⁠༽⁠つ Thank You!</b></div>
